# Three-statement linked model

Linked **P&L**, **balance sheet**, and **cash flow** in one `ModelBuilder` graph: operating performance, working capital, capex, and financing flow into cash and the balance sheet via DSL formulas.

## Concept

The statements crate evaluates a **directed acyclic graph** of nodes by period. Use `lag()` for prior-period balances, explicit nodes for WC **deltas** when you want stable first periods, and `compute()` for cross-statement links (for example retained earnings roll-forward and cash bridge). Export wide tables with `StatementResult.to_polars_wide()` — rows are `period_id`, columns are node ids.

**Note on linkage:** In this demo, `total_debt` and `common_equity` are **hardcoded value nodes** — they don't update dynamically from cash flow activity. The balance sheet balances because these values were manually calibrated. In a production model, you would compute debt from `coalesce(lag(total_debt, 1), debt_seed) - debt_repayment + debt_issuance` and similarly for equity, so that any P&L or CF assumption change automatically flows through to the balance sheet.

In [1]:
import json

from finstack.statements import Evaluator, ModelBuilder
from finstack.statements_analytics import explain_formula, trace_dependencies

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]

def series(vals):
    return list(zip(PERIODS, vals))

b = ModelBuilder("demo-3stmt")
b.periods("2025Q1..Q4", "2025Q2")

# --- P&L drivers ---
b.value("revenue", series([100.0, 105.0, 110.0, 118.0]))
b.value("cogs_pct", series([0.52, 0.51, 0.50, 0.49]))
b.compute("cogs", "revenue * cogs_pct")
b.compute("gross_profit", "revenue - cogs")
b.value("opex", series([22.0, 23.0, 24.0, 25.0]))
b.value("da", series([5.0, 5.0, 6.0, 6.0]))
b.compute("ebit", "gross_profit - opex - da")
b.value("interest_expense", series([3.0, 3.0, 3.2, 3.2]))
b.compute("ebt", "ebit - interest_expense")
b.value("tax_rate", series([0.25, 0.25, 0.25, 0.25]))
b.compute("taxes", "max(ebt, 0) * tax_rate")
b.compute("net_income", "ebt - taxes")

# --- Working capital (explicit deltas avoid NaN on first lag) ---
b.value("ar", series([18.0, 19.0, 20.0, 22.0]))
b.value("inventory", series([12.0, 12.5, 13.0, 14.0]))
b.value("ap", series([9.0, 9.5, 10.0, 11.0]))
b.value("delta_ar", series([0.0, 1.0, 1.0, 2.0]))
b.value("delta_inventory", series([0.0, 0.5, 0.5, 1.0]))
b.value("delta_ap", series([0.0, 0.5, 0.5, 1.0]))

# --- Capex & financing ---
b.value("capex", series([4.0, 4.5, 5.0, 5.5]))
b.value("debt_issuance", series([0.0, 0.0, 5.0, 0.0]))
b.value("debt_repayment", series([2.0, 2.0, 2.0, 2.0]))
b.value("equity_injection", series([0.0, 0.0, 0.0, 3.0]))

# --- Cash flow statement ---
b.compute("cfo", "net_income + da - delta_ar - delta_inventory + delta_ap")
b.compute("cfi", "- capex")
b.compute("cff", "debt_issuance - debt_repayment + equity_injection")
b.compute("net_change_cash", "cfo + cfi + cff")

# Opening cash via coalesce so period 1 seeds and later periods roll forward
b.value("cash_seed", series([25.0, 25.0, 25.0, 25.0]))
b.compute("cash", "coalesce(lag(cash, 1), cash_seed) + net_change_cash")

# PP&E roll-forward
b.value("ppe_seed", series([40.0, 40.0, 40.0, 40.0]))
b.compute("ppe_net", "coalesce(lag(ppe_net, 1), ppe_seed) + capex - da")

b.compute("total_assets", "cash + ar + inventory + ppe_net")

b.value("total_debt", series([30.0, 28.0, 31.0, 29.0]))
b.value("common_equity", series([50.0, 50.0, 50.0, 53.0]))
b.value("re_seed", series([4.0, 4.0, 4.0, 4.0]))
b.compute("retained_earnings", "coalesce(lag(retained_earnings, 1), re_seed) + net_income")
b.compute("total_liab_equity", "ap + total_debt + common_equity + retained_earnings")

spec = b.build()
result = Evaluator().evaluate(spec)

print("Built model:", spec.id, "| nodes:", spec.node_count, "| periods:", spec.period_count)
print("NI 2025Q3:", result.get("net_income", "2025Q3"))
print("Cash 2025Q4:", result.get("cash", "2025Q4"))
print("Total assets 2025Q4:", result.get("total_assets", "2025Q4"))

for p in PERIODS:
    ta = result.get("total_assets", p)
    tle = result.get("total_liab_equity", p)
    assert ta is not None and tle is not None
    assert abs(ta - tle) < 0.01, f"Balance sheet imbalance {p}: assets={ta} liab+eq={tle}"
print("Balance sheet balances (total_assets ≈ total_liab_equity) for all periods.")


Loaded finstack.statements + finstack.statements_analytics
Built model: demo-3stmt | nodes: 36 | periods: 4
NI 2025Q3: 16.35
Cash 2025Q4: 88.6725
Total assets 2025Q4: 161.6725
Balance sheet balances (total_assets ≈ total_liab_equity) for all periods.


In [2]:
import polars as pl
print("Polars OK:", pl.__version__)

PL_NODES = [
    "revenue",
    "cogs",
    "gross_profit",
    "opex",
    "da",
    "ebit",
    "interest_expense",
    "ebt",
    "taxes",
    "net_income",
]
BS_NODES = ["cash", "ar", "inventory", "ppe_net", "total_assets", "ap", "total_debt", "retained_earnings", "total_liab_equity"]
CF_NODES = ["cfo", "cfi", "cff", "net_change_cash"]

wide = result.to_polars_wide()
print("--- P&L (polars wide subset) ---")
print(wide.select(["period_id"] + [c for c in PL_NODES if c in wide.columns]))
print("--- Balance sheet ---")
print(wide.select(["period_id"] + [c for c in BS_NODES if c in wide.columns]))
print("--- Cash flow ---")
print(wide.select(["period_id"] + [c for c in CF_NODES if c in wide.columns]))

model_json = spec.to_json()
eval_json = result.to_json()
print("--- trace_dependencies(net_income) ---")
print(trace_dependencies(model_json, "net_income"))
print("--- explain_formula(net_income, 2025Q3) keys ---")
ex = explain_formula(model_json, eval_json, "net_income", "2025Q3")
print(sorted(ex.keys()))
print("final_value:", ex.get("final_value"))


Polars OK: 1.38.1
--- P&L (polars wide subset) ---
shape: (4, 11)
┌───────────┬─────────┬───────┬──────────────┬───┬──────────────────┬───────┬────────┬────────────┐
│ period_id ┆ revenue ┆ cogs  ┆ gross_profit ┆ … ┆ interest_expense ┆ ebt   ┆ taxes  ┆ net_income │
│ ---       ┆ ---     ┆ ---   ┆ ---          ┆   ┆ ---              ┆ ---   ┆ ---    ┆ ---        │
│ str       ┆ f64     ┆ f64   ┆ f64          ┆   ┆ f64              ┆ f64   ┆ f64    ┆ f64        │
╞═══════════╪═════════╪═══════╪══════════════╪═══╪══════════════════╪═══════╪════════╪════════════╡
│ 2025Q1    ┆ 100.0   ┆ 52.0  ┆ 48.0         ┆ … ┆ 3.0              ┆ 18.0  ┆ 4.5    ┆ 13.5       │
│ 2025Q2    ┆ 105.0   ┆ 53.55 ┆ 51.45        ┆ … ┆ 3.0              ┆ 20.45 ┆ 5.1125 ┆ 15.3375    │
│ 2025Q3    ┆ 110.0   ┆ 55.0  ┆ 55.0         ┆ … ┆ 3.2              ┆ 21.8  ┆ 5.45   ┆ 16.35      │
│ 2025Q4    ┆ 118.0   ┆ 57.82 ┆ 60.18        ┆ … ┆ 3.2              ┆ 25.98 ┆ 6.495  ┆ 19.485     │
└───────────┴─────────┴───────┴───

## Takeaways

- One evaluator run materializes **P&L**, **BS**, and **CF** together; `lag()` ties periods.
- Use **explicit delta nodes** when you need clean first periods; otherwise `lag()` can yield `NaN` at the boundary.
- `to_polars_wide()` is convenient for **slice-and-plot** workflows; pair with `trace_dependencies` / `explain_formula` for audit trails.